In [ ]:
import threading
import time

import cv2
import requests

# Configuration
API_URL = "https://predict.ultralytics.com"
HEADERS = {"x-api-key": "3c22078e18ed4bd513954eb096328f8465aca49109"}
DATA = {
    "model": "https://hub.ultralytics.com/models/aFMHXlwhCJY2QgS28Zs4",
    "imgsz": 640,
    "conf": 0.25,
    "iou": 0.45,
}

# Global variables for thread communication
latest_frame = None
detections = []
is_running = True


def inference_worker():
    """Background thread that talks to the API."""
    global detections, latest_frame, is_running

    while is_running:
        if latest_frame is not None:
            # 1. Prepare the image from the latest frame
            _, img_encoded = cv2.imencode(".jpg", latest_frame)
            img_bytes = img_encoded.tobytes()

            try:
                # 2. API Request
                response = requests.post(
                    API_URL,
                    headers=HEADERS,
                    data=DATA,
                    files={"file": ("image.jpg", img_bytes, "image/jpeg")},
                    timeout=5,  # Don't hang forever
                )
                results = response.json()

                # 3. Parse and Update Global Detections
                new_detections = []
                if isinstance(results, list):
                    new_detections = results
                elif isinstance(results, dict) and "images" in results:
                    new_detections = results["images"][0].get("results", [])

                detections = new_detections

            except Exception as e:
                print(f"Inference Error: {e}")

        # Small sleep to prevent spamming the API too fast (helps with quota)
        time.sleep(0.1)


# Start the background thread
thread = threading.Thread(target=inference_worker, daemon=True)
thread.start()

# Initialize Webcam
cap = cv2.VideoCapture(0)

print("Starting Real-time video. Labels will update as fast as internet allows...")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Update the frame for the worker thread
    latest_frame = frame.copy()

    # Draw the CURRENT detections (which might be from a few frames ago)
    for det in detections:
        box = det.get("box")
        if box:
            x1, y1 = int(box.get("x1", 0)), int(box.get("y1", 0))
            x2, y2 = int(box.get("x2", 0)), int(box.get("y2", 0))

            label = f"{det.get('name', 'unknown')} {det.get('conf', 0.0):.2f}"

            # Draw on the live frame
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(
                frame,
                label,
                (x1, y1 - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (0, 255, 0),
                2,
            )

    # Show the LIVE frame (No waiting for API!)
    cv2.imshow("Real-time Video (Delayed Labels)", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        is_running = False
        break

cap.release()
cv2.destroyAllWindows()

Starting Real-time video. Labels will update as fast as internet allows...


KeyboardInterrupt: 